In [1]:
import os
os.environ['R_HOME'] = '/Users/soeren.becker/micromamba/envs/env_pertpy/lib/R'

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import pertpy as pt
import scanpy as sc
import anndata as ad
import seaborn as sns
import decoupler as dc
import matplotlib.pyplot as plt

from pathlib import Path
from rpy2 import robjects as ro
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import get_conversion, localconverter
from rpy2.robjects.packages import importr
from scipy.sparse import issparse
from tqdm.auto import tqdm

/Users/soeren.becker/micromamba/envs/env_pertpy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [3]:
edger = importr("edgeR")

In [4]:
CELLTYPE = "CD14_Mono"

paths = sorted(glob.glob(f"/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks/pbulk_{CELLTYPE}_*.h5ad"))

print(len(paths))

91


In [11]:
def combine_number_of_cell_per_donor(adata1, adata2):
    shared_idcs = adata1.obs.index.intersection(adata2.obs.index)

    unique_idcs1 = adata1.obs.index[~np.isin(adata1.obs.index, adata2.obs.index)]
    unique_idcs2 = adata2.obs.index[~np.isin(adata2.obs.index, adata1.obs.index)]
    
    adata1_shared = adata1[shared_idcs]
    adata2_shared = adata2[shared_idcs]
    
    combined = (adata1_shared.obs.psbulk_n_cells + adata2_shared.obs.psbulk_n_cells).to_frame()
    new_idcs = [*shared_idcs]
    for _idx1 in unique_idcs1:
        combined.loc[_idx1, "psbulk_n_cells"] = adata1.obs.loc[_idx1].psbulk_n_cells
        new_idcs.append(_idx1)
    
    for _idx2 in unique_idcs2:
        combined.loc[_idx2, "psbulk_n_cells"] = adata2.obs.loc[_idx2].psbulk_n_cells
        new_idcs.append(_idx2)

    return combined, new_idcs

def combine_counts(adata1, adata2):
    assert np.all(adata1.var_names == adata2.var_names)
    shared_idcs = adata1.obs.index.intersection(adata2.obs.index)
    unique_idcs1 = adata1.obs.index[~np.isin(adata1.obs.index, adata2.obs.index)]
    unique_idcs2 = adata2.obs.index[~np.isin(adata2.obs.index, adata1.obs.index)]
    adata1_shared = adata1[shared_idcs]
    adata2_shared = adata2[shared_idcs]
    assert np.all(adata1_shared.obs.index == adata2_shared.obs.index)
    
    X = adata1_shared.X + adata2_shared.X
    new_idcs = [*shared_idcs]
    if len(unique_idcs1) > 0:
        X = np.vstack([X, adata1[unique_idcs1].X])
        new_idcs.extend(unique_idcs1)
    if len(unique_idcs2) > 0:
        X = np.vstack([X, adata2[unique_idcs2].X])
        new_idcs.extend(unique_idcs2)
        
    return X, new_idcs

def combine_frac_expression_cells_total(adata1, adata2):
    assert all(adata1.var.index == adata2.var.index)
    n1 = adata1.obs.psbulk_n_cells.sum()
    n2 = adata2.obs.psbulk_n_cells.sum()
    f1 = adata1.var.frac_expressing_cells_total
    f2 = adata2.var.frac_expressing_cells_total
    m1 = n1 * f1
    m2 = n2 * f2
    return ((m1+m2) / (n1+n2)).to_frame()

def combine_pseudobulks(adata1, adata2, new_celltype_label):
    X, new_idcs_counts = combine_counts(adata1, adata2)
    num_cells, new_idcs_num_cells = combine_number_of_cell_per_donor(adata1, adata2)
    assert new_idcs_counts == new_idcs_num_cells
    frac_expressing_cells = combine_frac_expression_cells_total(adata1, adata2)

    obs = num_cells
    obs.loc[:, "celltype"] = new_celltype_label
    
    shared_idcs = adata1.obs.index.intersection(adata2.obs.index)
    unique_idcs1 = adata1.obs.index[~np.isin(adata1.obs.index, adata2.obs.index)]
    unique_idcs2 = adata2.obs.index[~np.isin(adata2.obs.index, adata1.obs.index)]

    
    donors = adata1[shared_idcs].obs.donor
    for _idx1 in unique_idcs1:
        donors.loc[_idx1] = adata1.obs.loc[_idx1].donor
    for _idx2 in unique_idcs2:
        donors.loc[_idx2] = adata2.obs.loc[_idx2].donor
    obs.loc[:, "donor"] = donors

    if not "bc1_well" in adata1.obs.columns:
        adata1.obs.loc[:, "bc1_well"] = np.nan

    if not "bc1_well" in adata2.obs.columns:
        adata1.obs.loc[:, "bc1_well"] = np.nan
    
    bc1_well = adata1[shared_idcs].obs.bc1_well
    for _idx1 in unique_idcs1:
        bc1_well.loc[_idx1] = adata1.obs.loc[_idx1].bc1_well
    for _idx2 in unique_idcs2:
        bc1_well.loc[_idx2] = adata2.obs.loc[_idx2].bc1_well
    obs.loc[:, "bc1_well"] = bc1_well

    cytokine1 = adata1.obs.cytokine.unique()
    cytokine2 = adata2.obs.cytokine.unique()
    
    assert len(cytokine1) == 1
    assert len(cytokine2) == 1
    assert cytokine1[0] == cytokine2[0]
    obs.loc[:, "cytokine"] = cytokine1[0]

    adata_sum = ad.AnnData(
        X = X,
        obs=obs,
        var=frac_expressing_cells,
    )
    return adata_sum

In [15]:
def load_and_combine_monocytes(path):
    pdata1, pdata_pbs1, cytokine1 = load_data(path)
    pdata2, pdata_pbs2, cytokine2 = load_data(path.replace("CD14", "CD16"))
    assert cytokine1 == cytokine2
    pdata = combine_pseudobulks(pdata1, pdata2, "Mono")
    pdata_pbs = combine_pseudobulks(pdata_pbs1, pdata_pbs2, "Mono")
    return pdata, pdata_pbs, cytokine1

def load_data(path):
    pdata = sc.read_h5ad(path)
    cytokine = pdata.obs.cytokine.unique()
    assert len(cytokine) == 1, cytokine
    cytokine = cytokine[0]
    celltype = (Path(path).stem).replace("pbulk_", "").replace("_"+cytokine, "")
    pdata_pbs = sc.read_h5ad(os.path.join(str(Path(path).parent), f"pbulk_{celltype}_pbs.h5ad"))    
    return pdata, pdata_pbs, cytokine

def apply_filters_and_concat(pdata, pdata_pbs, frac_cells=0.05, min_cells=10):
    # filter genes
    pdata, pdata_pbs = filter_genes(pdata, pdata_pbs, frac_cells=frac_cells)
    # assert that genes are the same
    assert np.all(pdata.var_names == pdata_pbs.var_names)
    if pdata.shape[1] == 0:
        print(f"{cytokine}: No valid genes for cytokine.")
        return None

    # filter pseudobulk samples
    pdata, pdata_pbs = filter_samples(pdata, pdata_pbs, min_cells=min_cells)
    # assert that samples are paired
    d_pbs = pdata_pbs.obs.donor.unique()
    d_cyto = pdata.obs.donor.unique()
    assert len(np.intersect1d(d_pbs, d_cyto)) == len(d_pbs)
    # check that there are enough samples
    if pdata.shape[0] < 2:
        # we need more than 1 to estimate dispersion
        print(f"{cytokine}: insufficient number of cytokine-treated pseudobulk samples pdata.shape={pdata.shape}.")
        return None
    if pdata_pbs.shape[0] < 2:
        # we need more than 1 to estimate dispersion
        print(f"{cytokine}: insufficient number of PBS-treated pseudobulk samples pdata_pbs.shape={pdata_pbs.shape}.")
        return None
    
    return ad.concat([pdata, pdata_pbs])

def filter_genes(pdata, pdata_pbs, frac_cells=0.05):
    # we keep a gene if it is expressed by at least 5 % of control cells OR 5 % of cytokine-treated cells
    gene_mask = (pdata.var.frac_expressing_cells_total >= frac_cells) | (pdata_pbs.var.frac_expressing_cells_total >= frac_cells)
    valid_genes = pdata.var_names[gene_mask]
    return pdata[:, gene_mask].copy(), pdata_pbs[:, gene_mask].copy()

def filter_samples(pdata, pdata_pbs, min_cells=10):
    # keep donors whose pseudobulks consist of at least 10 cells in both PBS AND cytokine condition
    valid_donors_1 = pdata.obs.loc[pdata.obs.psbulk_n_cells >= min_cells].donor.values
    valid_donors_2 = pdata_pbs.obs.loc[pdata_pbs.obs.psbulk_n_cells >= min_cells].donor.values
    valid_donors = np.intersect1d(valid_donors_1, valid_donors_2)
    return pdata[pdata.obs.donor.isin(valid_donors)].copy(), pdata_pbs[pdata_pbs.obs.donor.isin(valid_donors)].copy()

def run_de_test(pdata, contrast: str, design=None):
    if design is None:
        design = "~donor+cytokine"
    edgr = pt.tl.EdgeR(pdata, design=design)
    edgr.fit()
    res_df = edgr.test_contrasts(
        edgr.contrast(
            column="cytokine",
            baseline="PBS",
            group_to_compare=contrast,
        )
    )
    return res_df

In [16]:
for path in tqdm(paths):
    pdata, pdata_pbs, cytokine = load_and_combine_monocytes(path)
    if cytokine == "PBS":
        continue
    pdata.write_h5ad(path.replace("CD14_Mono", "Mono"))
    # break

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [00:42<00:00,  2.15it/s]


In [186]:
all_dfs = []

CELLTYPE = "Mono"

for k, path in enumerate(tqdm(paths)):

    # load and filter
    # pdata, pdata_pbs, cytokine = load_data(path)
    pdata, pdata_pbs, cytokine = load_and_combine_monocytes(path)
    if cytokine == "PBS":
        continue

    # apply filters on genes and samples, concat pbs and cytokine
    pdata_both = apply_filters_and_concat(pdata, pdata_pbs, frac_cells=0.05, min_cells=10)
    if pdata_both is None:
        # insufficient number of genes or samples
        continue
    
    # run test
    df = run_de_test(pdata_both, design="~donor+cytokine", contrast=cytokine)
    df.loc[:, "celltype"] = CELLTYPE
    df.loc[:, "cytokine"] = cytokine
    # df.loc[:, "num_donors"] = pdata_both.obs.donor.nunique()
    # df.loc[:, "median_num_cells_pbs"] = pdata_both.obs.loc[pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.median()
    # df.loc[:, "median_num_cells_cytokine"] = pdata_both.obs.loc[pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.median()
    # df.loc[:, "mean_num_cells_pbs"] = pdata_both.obs.loc[pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.mean()
    # df.loc[:, "mean_num_cells_cytokine"] = pdata_both.obs.loc[pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.mean()
    # df.loc[:, "min_num_cells_pbs"] = pdata_both.obs.loc[pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.min()
    # df.loc[:, "min_num_cells_cytokine"] = pdata_both.obs.loc[pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.min()
    # df.loc[:, "max_num_cells_pbs"] = pdata_both.obs.loc[pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.max()
    # df.loc[:, "max_num_cells_cytokine"] = pdata_both.obs.loc[pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.max()
    all_dfs.append(df)

result_df = pd.concat(all_dfs, axis=0)
result_df.to_csv(f"/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_{CELLTYPE}.csv")

  0%|                                                                                                                                                                                | 0/91 [00:00<?, ?it/s]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


  1%|█▊                                                                                                                                                                      | 1/91 [00:03<05:40,  3.78s/it]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


  2%|███▋                                                                                                                                                                    | 2/91 [00:07<05:23,  3.64s/it]

...

 98%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎   | 89/91 [05:07<00:06,  3.36s/it]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏ | 90/91 [05:10<00:03,  3.33s/it]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [05:13<00:00,  3.45s/it]


In [187]:
result_df.to_csv(f"/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_Mono_Oct.csv")

In [188]:
result_df

,variable,log_fc,logCPM,F,p_value,adj_p_value,contrast,celltype,cytokine
0,ACTBP11,1.692541,2.654259,5.260664e+02,2.195401e-13,2.008133e-09,None,Mono,4-1BBL
1,ARPC4,0.582826,7.010982,4.345288e+02,9.248392e-13,4.229752e-09,None,Mono,4-1BBL
2,ENSG00000289901,-1.042720,3.573044,3.221062e+02,8.632017e-12,2.631902e-08,None,Mono,4-1BBL
3,ACTG1,0.615921,8.829680,2.425726e+02,6.973510e-11,1.385970e-07,None,Mono,4-1BBL
4,AP2M1,0.398562,6.112964,2.398383e+02,7.576092e-11,1.385970e-07,None,Mono,4-1BBL
...,...,...,...,...,...,...,...,...,...
9284,SFI1,-0.013976,5.510068,1.454970e-01,9.987591e-01,9.991893e-01,None,Mono,VEGF
9285,BMT2,-0.000039,7.221322,8.910246e-07,9.992591e-01,9.995820e-01,None,Mono,VEGF
9286,GARRE1,-0.001196,4.765221,1.156293e-03,9.995536e-01,9.997110e-01,None,Mono,VEGF
9287,ARL3,-0.000029,3.731223,2.553710e-07,9.996034e-01,9.997110e-01,None,Mono,VEGF
